# Bearing fault diagnosis — Colab run

Self-contained: no project files needed. Run the cells in order.

Downloads the Paderborn bearing data, checks the file layout, builds the per-window feature table and saves it to Google Drive.

## 1. Setup and settings

Settings. Raw data goes to Colab's temporary disk; the feature table is saved to Drive.

In [ ]:
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null

import shutil
import subprocess
import tempfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import butter, hilbert, sosfiltfilt
from scipy.stats import kurtosis, skew

RAW_DIR = Path("/content/paderborn_raw")  # Colab temporary disk
OUT_DIR = Path("/content/drive/MyDrive/load_bearing_research_paper/data/features")  # kept in Drive

CONDITION = "N15_M07_F10"      # 1500 rpm, 0.7 Nm, 1000 N
FS = 64_000                    # vibration sampling rate (Hz)
SHAFT_HZ = 1500 / 60           # shaft speed (Hz)
RECORDINGS = 20                # 4-second recordings per bearing
WINDOW = int(1.0 * FS)         # 1-second windows
BASE_URL = "https://groups.uni-paderborn.de/kat/BearingDataCenter/"

## 2. Mount Google Drive (to keep the feature table)

Mount Google Drive so the feature table survives the session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Bearings and fault frequencies

In [ ]:
HEALTHY, INNER, OUTER = "healthy", "inner_race", "outer_race"

BEARINGS = pd.DataFrame(
    [(c, HEALTHY, "none") for c in ["K001", "K002", "K003", "K004", "K005", "K006"]]
    + [(c, OUTER, "artificial") for c in ["KA01", "KA03", "KA05", "KA06", "KA07", "KA08", "KA09"]]
    + [(c, INNER, "artificial") for c in ["KI01", "KI03", "KI05", "KI07", "KI08"]]
    + [(c, OUTER, "real") for c in ["KA04", "KA15", "KA16", "KA22", "KA30"]]
    + [(c, INNER, "real") for c in ["KI04", "KI14", "KI16", "KI17", "KI18", "KI21"]],
    columns=["bearing", "label", "origin"],
)

# Bearing 6203: 8 balls, ball diameter 6.75 mm, pitch diameter 28.55 mm
N_BALLS, BALL_D, PITCH_D = 8, 6.75, 28.55
RATIO = BALL_D / PITCH_D
BPFO = N_BALLS / 2 * (1 - RATIO) * SHAFT_HZ   # ~76.4 Hz
BPFI = N_BALLS / 2 * (1 + RATIO) * SHAFT_HZ   # ~123.6 Hz
print(f"{len(BEARINGS)} bearings | BPFO = {BPFO:.1f} Hz | BPFI = {BPFI:.1f} Hz")
print(BEARINGS.groupby(["origin", "label"]).size())

## 4. Download and extract (about 4.6 GB, 10-20 minutes)

Downloads the 29 study bearings (~160 MB each, ~4.6 GB total) from the Paderborn KAt-DataCenter and keeps only the `N15_M07_F10` recordings. Takes 10–20 minutes; re-running skips bearings already present.

Data licence: CC BY-NC 4.0 — cite Lessmeier et al. (2016) and the KAt-DataCenter.

In [ ]:
def recording_path(code, n):
    return RAW_DIR / code / f"{CONDITION}_{code}_{n}.mat"


def is_complete(code):
    return all(recording_path(code, n).exists() for n in range(1, RECORDINGS + 1))


def fetch_bearing(code):
    """Download one bearing archive, keep only this condition's files, delete the archive."""
    with tempfile.TemporaryDirectory() as tmp:
        archive = Path(tmp) / f"{code}.rar"
        with urllib.request.urlopen(BASE_URL + f"{code}.rar") as r, open(archive, "wb") as f:
            shutil.copyfileobj(r, f, length=1 << 20)
        out = Path(tmp) / "x"
        out.mkdir()
        subprocess.run(["unrar", "x", "-o+", "-inul", str(archive), f"*{CONDITION}*", f"{out}/"], check=False)
        files = sorted(out.rglob(f"{CONDITION}_{code}_*.mat"))
        if not files:  # filter missed files in subfolders: extract everything instead
            subprocess.run(["unrar", "x", "-o+", "-inul", str(archive), f"{out}/"], check=True)
            files = sorted(out.rglob(f"{CONDITION}_{code}_*.mat"))
        (RAW_DIR / code).mkdir(parents=True, exist_ok=True)
        for p in files:
            shutil.move(str(p), RAW_DIR / code / p.name)
        return len(files)


def download_all():
    for code in BEARINGS.bearing:
        if is_complete(code):
            print(f"{code}: already present")
            continue
        try:
            print(f"{code}: {fetch_bearing(code)} files")
        except Exception as e:
            print(f"{code}: FAILED - {e}")
    missing = [c for c in BEARINGS.bearing if not is_complete(c)]
    print("Incomplete bearings:", missing or "none")

download_all()

## 5. Load a recording and check the file layout

Should list `vibration_1` among the channels and show about 256,000 samples (4 s).

In [ ]:
def channel_name(ch):
    name = ch["Name"]
    while isinstance(name, np.ndarray):
        name = name.flat[0]
    return str(name)


def load_vibration(path, channel="vibration_1"):
    record = loadmat(path)[Path(path).stem][0, 0]
    for ch in record["Y"].ravel():
        if channel_name(ch) == channel:
            return np.asarray(ch["Data"], dtype=float).ravel()
    raise KeyError(f"{channel} not in {[channel_name(c) for c in record['Y'].ravel()]}")


check = recording_path("KA01", 1)
record = loadmat(check)[check.stem][0, 0]
print("Channels:", [channel_name(ch) for ch in record["Y"].ravel()])
x = load_vibration(check)
print(f"{len(x)} samples = {len(x) / FS:.2f} s")

## 6. Features

Time-domain statistics and envelope-spectrum energy at the outer-race (BPFO) and inner-race (BPFI) fault frequencies.

In [ ]:
def time_features(x):
    rms = np.sqrt(np.mean(x**2))
    return {
        "rms": rms,
        "kurtosis": kurtosis(x, fisher=False),
        "skewness": skew(x),
        "crest_factor": np.max(np.abs(x)) / rms if rms > 0 else 0.0,
        "peak_to_peak": np.ptp(x),
    }


SOS = butter(4, [2_000, 10_000], btype="bandpass", fs=FS, output="sos")


def envelope_spectrum(x):
    env = np.abs(hilbert(sosfiltfilt(SOS, x)))
    env -= env.mean()
    return np.fft.rfftfreq(len(env), 1 / FS), np.abs(np.fft.rfft(env)) / len(env)


def fault_features(x, harmonics=2, tol=0.02):
    """Peak near k*BPFO / k*BPFI divided by the envelope-spectrum median (10-500 Hz)."""
    freqs, amps = envelope_spectrum(x)
    floor = np.median(amps[(freqs >= 10) & (freqs <= 500)]) or 1e-12
    out = {}
    for name, f0 in {"bpfo": BPFO, "bpfi": BPFI}.items():
        for k in range(1, harmonics + 1):
            band = (freqs >= k * f0 * (1 - tol)) & (freqs <= k * f0 * (1 + tol))
            out[f"{name}_h{k}"] = amps[band].max() / floor
    return out


def window_features(x):
    return {**time_features(x), **fault_features(x)}

## 7. Build the feature table and save it

About 2,300 windows (29 bearings × 20 recordings × 4 windows).

In [ ]:
def build_features():
    rows = []
    for b in BEARINGS.itertuples():
        for n in range(1, RECORDINGS + 1):
            path = recording_path(b.bearing, n)
            try:
                signal = load_vibration(path)
            except Exception as e:
                print(f"skipped {path.name}: {e}")
                continue
            for w in range(len(signal) // WINDOW):
                x = signal[w * WINDOW:(w + 1) * WINDOW]
                rows.append({"bearing": b.bearing, "label": b.label, "origin": b.origin,
                             "recording": n, "window": w, **window_features(x)})
        print(f"{b.bearing} done")
    return pd.DataFrame(rows)


features = build_features()
out_file = OUT_DIR / f"features_{CONDITION}.csv"
features.to_csv(out_file, index=False)
print(f"{len(features)} windows saved to {out_file}")

## 8. Sanity check

Outer-race bearings should show higher `bpfo_h1`, inner-race bearings higher `bpfi_h1`, healthy bearings near 1. Compare how much weaker the pattern is for real damage than for artificial damage.

In [ ]:
print(features.groupby(["origin", "label"]).bearing.nunique(), "\n")
features.groupby(["origin", "label"])[["kurtosis", "bpfo_h1", "bpfi_h1"]].median().round(2)